# Deep Ensemble with DNABERT

This notebook fine-tunes a 5-member deep ensemble of DNABERT models with LoRA adapters on GUIDE-seq training data, runs inference on both GUIDE-seq (in-distribution) and CHANGE-seq (out-of-distribution) test sets, and extracts DNABERT embeddings for drift detection. Each ensemble member is trained independently with a different random seed to ensure diverse uncertainty estimates.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Dataset, Subset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils import resample
from scipy.special import expit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Ti


## Data Loading

In [2]:
DATA_DIR = 'data/processed/'

# Loading raw sequences for DNABERT tokenisation
guide_train = pd.read_csv(f'{DATA_DIR}guide_train.csv', low_memory=False)
guide_test = pd.read_csv(f'{DATA_DIR}guide_test.csv', low_memory=False)
change_train = pd.read_csv(f'{DATA_DIR}change_train.csv', low_memory=False)
change_test = pd.read_csv(f'{DATA_DIR}change_test.csv', low_memory=False)

print(f'GUIDE-seq train: {guide_train.shape}, test: {guide_test.shape}')
print(f'CHANGE-seq train: {change_train.shape}, test: {change_test.shape}')

GUIDE-seq train: (1229148, 12), test: (248614, 12)
CHANGE-seq train: (2352636, 10), test: (520991, 10)


## DNABERT Tokenisation

In [3]:
GRNA_COL = 'target'
TARGET_COL = 'offtarget_sequence'
DNABERT_MODEL = 'zhihan1996/DNA_bert_3'

# Load tokeniser
tokenizer = AutoTokenizer.from_pretrained(
    DNABERT_MODEL,
    trust_remote_code=True
)
print(f'Tokeniser loaded: {DNABERT_MODEL}')

Tokeniser loaded: zhihan1996/DNA_bert_3


In [4]:
K = 3

# Converts a DNA sequence into overlapping 3-mers for DNABERT input
# e.g. 'ATCG' -> 'ATC TCG'
def sequence_to_kmers(seq, k=K):
    return ' '.join(seq[i:i+k] for i in range(len(seq) - k + 1))

In [5]:
# Tokenises sequences on the fly instead of storing the full encoded dataset
class CRISPRDataset(Dataset):
    def __init__(self, df, grna_col, target_col, tokenizer, max_length=45):
        self.df = df.reset_index(drop=True)
        self.grna_col = grna_col
        self.target_col = target_col
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = torch.tensor(self.df['label'].values, dtype=torch.float32)
        # Converts to lists for fast indexing during training
        self.grna_seqs = self.df[grna_col].tolist()
        self.target_seqs = self.df[target_col].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Converts sequences to 3-mers on-the-fly
        grna_kmers = sequence_to_kmers(self.grna_seqs[idx])
        target_kmers = sequence_to_kmers(self.target_seqs[idx])

        encoded = self.tokenizer(
            grna_kmers,
            target_kmers,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'token_type_ids': encoded['token_type_ids'].squeeze(0),
            'label': self.labels[idx]
        }

# Quick check
dataset = CRISPRDataset(guide_train, GRNA_COL, TARGET_COL, tokenizer)
print(f'Dataset size: {len(dataset)}')
sample = dataset[0]
print(f'input_ids shape: {sample["input_ids"].shape}')
print(f'attention_mask shape: {sample["attention_mask"].shape}')
print(f'token_type_ids shape: {sample["token_type_ids"].shape}')
print(f'label: {sample["label"]}')

Dataset size: 1229148
input_ids shape: torch.Size([45])
attention_mask shape: torch.Size([45])
token_type_ids shape: torch.Size([45])
label: 1.0


## LoRA Configuration and Model

In [6]:
# DNABERT encoder with a binary classification head
class DNABERTClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        self.dropout = nn.Dropout(0.1)
        # Binary classification head
        self.classifier = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_embedding))
        return logits.squeeze(-1)

# LoRA config (parameter-efficient adaptation to reduce trainable parameters and memory requirements)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.FEATURE_EXTRACTION
)

# Fresh backbone load for each call to ensure independent LoRA initialisations across ensemble members
def build_lora_model():
    model = DNABERTClassifier(DNABERT_MODEL)
    model.backbone = get_peft_model(model.backbone, lora_config)
    return model

# Test build
model = build_lora_model()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Checking trainable parameters in the LoRA backbone and full model
model.backbone.print_trainable_parameters()

trainable = 0
total = 0

for p in model.parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print(f'Total trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)')
print(f'Classifier trainable: {model.classifier.weight.requires_grad}')

trainable params: 294,912 || all params: 86,389,248 || trainable%: 0.3414
Total trainable: 295,681 / 86,390,017 (0.34%)
Classifier trainable: True


## Training Loop

In [8]:
def train_member(model, train_dataset, epochs=3, batch_size=16, lr=2e-5, seed=42):
    torch.manual_seed(seed)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    
    # Imbalance handled via undersampling
    criterion = nn.BCEWithLogitsLoss()
    
    model.to(device)
    
    # Passing only trainable parameters (LoRA adapters + classifier head)
    optimiser = AdamW((p for p in model.parameters() if p.requires_grad), lr=lr)
    
    num_training_steps = len(train_loader) * epochs
    num_warmup_steps = int(0.1 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['label'].to(device)
            
            optimiser.zero_grad()
            logits = model(input_ids, attention_mask, token_type_ids)
            loss = criterion(logits, labels)
            
            loss.backward()
            # Clip gradients for training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            scheduler.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}')
    
    return model

In [9]:
positives = guide_train[guide_train['label'] == 1]
negatives = guide_train[guide_train['label'] == 0]

# Keeps all positives and subsample negatives to a 10:1 ratio
n_neg_sampled = min(len(positives) * 10, len(negatives))

negatives_sub = resample(
    negatives,
    replace=False,
    n_samples=n_neg_sampled,
    random_state=2000
)

guide_train_sub = (
    pd.concat([positives, negatives_sub])
    .sample(frac=1, random_state=2000)
    .reset_index(drop=True)
)

# Records the actual source-domain negative sampling rate to be used later to correct probabilities for case-control undersampling
negative_sampling_rate = n_neg_sampled / len(negatives)
sampling_logit_offset = np.log(negative_sampling_rate)

print(f'Full GUIDE positives: {len(positives):,}')
print(f'Full GUIDE negatives: {len(negatives):,}')
print(f'Sampled negatives: {n_neg_sampled:,}')
print(f'Subsampled train size: {len(guide_train_sub):,}')
print(f'Label distribution:\n{guide_train_sub["label"].value_counts()}')
print(f'Negative sampling rate: {negative_sampling_rate:.8f}')
print(f'Logit correction offset: {sampling_logit_offset:.6f}')

# Saves training data for evaluation
np.savez(
    DATA_DIR + 'ensemble_training_metadata.npz',
    n_positive=len(positives),
    n_negative_total=len(negatives),
    n_negative_sampled=n_neg_sampled,
    negative_sampling_rate=negative_sampling_rate,
    sampling_logit_offset=sampling_logit_offset
)

guide_train_dataset = CRISPRDataset(
    guide_train_sub,
    GRNA_COL,
    TARGET_COL,
    tokenizer
)

Full GUIDE positives: 1,371
Full GUIDE negatives: 1,227,777
Sampled negatives: 13,710
Subsampled train size: 15,081
Label distribution:
label
0    13710
1     1371
Name: count, dtype: int64
Negative sampling rate: 0.01116652
Logit correction offset: -4.494835


In [10]:
# Ensemble Training
save_dir = 'models/'
os.makedirs(save_dir, exist_ok=True)

seeds = [1847, 9021, 337, 6154, 7288]
N_MEMBERS = len(seeds)

for i, seed in enumerate(seeds):
    print(f'\nTraining ensemble member {i+1}/{N_MEMBERS} (seed={seed})...')
    
    torch.manual_seed(seed)
    model = build_lora_model()
    model = train_member(model, guide_train_dataset, seed=seed)

    # Move to CPU before saving/freeing GPU
    model.to('cpu')
    
    # Save LoRA adapter and classifier separately
    member_dir = os.path.join(save_dir, f'ensemble_member_{i+1}')
    model.backbone.save_pretrained(member_dir)
    torch.save(model.classifier.state_dict(), os.path.join(member_dir, 'classifier.pt'))
    print(f'Member {i+1} saved to {member_dir}')
    
    # Free memory before next member
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print(f'Member {i+1} done')

print('\nAll ensemble members trained')


Training ensemble member 1/5 (seed=1847)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/3 - Loss: 0.4125
Epoch 2/3 - Loss: 0.2380
Epoch 3/3 - Loss: 0.2226
Member 1 saved to models/ensemble_member_1
Member 1 done

Training ensemble member 2/5 (seed=9021)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/3 - Loss: 0.4301
Epoch 2/3 - Loss: 0.2559
Epoch 3/3 - Loss: 0.2472
Member 4 saved to models/ensemble_member_4
Member 4 done

Training ensemble member 5/5 (seed=7288)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/3 - Loss: 0.3615
Epoch 2/3 - Loss: 0.2358
Epoch 3/3 - Loss: 0.2182
Member 5 saved to models/ensemble_member_5
Member 5 done

All ensemble members trained


## Ensemble Inference

In [11]:
save_dir = 'models/'
N_MEMBERS = 5

In [12]:
def load_ensemble(save_dir, n_members, model_name):
    ensemble = []
    for i in range(1, n_members + 1):
        member_dir = os.path.join(save_dir, f'ensemble_member_{i}')
        
        # Rebuilds model and loads saved LoRA adapter
        model = DNABERTClassifier(model_name)
        model.backbone = PeftModel.from_pretrained(
            model.backbone,
            member_dir,
            is_trainable=False
        )
        
        # Loads trained classifier head
        model.classifier.load_state_dict(
            torch.load(
                os.path.join(member_dir, 'classifier.pt'),
                map_location='cpu'
            )
        )
        
        model.eval()
        ensemble.append(model)
        print(f'Loaded member {i}')
    
    return ensemble

ensemble = load_ensemble(save_dir, N_MEMBERS, DNABERT_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded member 1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded member 4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded member 5


## Generating Ensemble Predictions

In [13]:
# Returns raw logits with shape (n_members, n_samples)
def ensemble_predict_logits(ensemble, dataset, batch_size=512):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    all_logits = []

    for i, model in enumerate(ensemble, 1):
        model.to(device)
        model.eval()
        member_logits = []

        with torch.inference_mode():
            for batch in loader:
                logits = model(
                    batch['input_ids'].to(device),
                    batch['attention_mask'].to(device),
                    batch['token_type_ids'].to(device)
                )

                member_logits.append(
                    logits.float().cpu().numpy()
                )

        all_logits.append(np.concatenate(member_logits))

        model.to('cpu')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f'Member {i}/{len(ensemble)} done')

    all_logits = np.stack(all_logits)

    assert all_logits.shape == (len(ensemble), len(dataset))

    return all_logits

In [14]:
# Creating test datasets
guide_test_dataset  = CRISPRDataset(guide_test,  GRNA_COL, TARGET_COL, tokenizer)
change_test_dataset = CRISPRDataset(change_test, GRNA_COL, TARGET_COL, tokenizer)

# Run inference - in-distribution
print('GUIDE-seq test...')
guide_logits = ensemble_predict_logits(ensemble, guide_test_dataset)
print('Done')

# Run inference - OOD
print('CHANGE-seq test...')
change_logits = ensemble_predict_logits(ensemble, change_test_dataset)
print('Done')

# Quick sanity check
print(f'\nguide_logits  {guide_logits.shape}')
print(f'change_logits {change_logits.shape}')

# Saves per-member logits
np.save(os.path.join(DATA_DIR, 'guide_member_logits.npy'), guide_logits)
np.save(os.path.join(DATA_DIR, 'change_member_logits.npy'), change_logits)
print('Saved per-member logits')

GUIDE-seq test...
Member 1/5 done
Member 2/5 done
Member 3/5 done
Member 4/5 done
Member 5/5 done
Done
CHANGE-seq test...
Member 1/5 done
Member 2/5 done
Member 3/5 done
Member 4/5 done
Member 5/5 done
Done

guide_logits  (5, 248614)
change_logits (5, 520991)
Saved per-member logits


In [15]:
# Convert per-member logits to probabilities
guide_member_probs = expit(guide_logits)
change_member_probs = expit(change_logits)

# Ensemble prediction is the mean member probability
guide_mean_probs = guide_member_probs.mean(axis=0)
change_mean_probs = change_member_probs.mean(axis=0)

y_guide_test = guide_test['label'].to_numpy()
y_change_test = change_test['label'].to_numpy()

# Sanity checks
assert guide_mean_probs.shape == y_guide_test.shape
assert change_mean_probs.shape == y_change_test.shape
assert np.isfinite(guide_member_probs).all()
assert np.isfinite(change_member_probs).all()

# Save fresh ensemble mean probabilities
np.save(DATA_DIR + 'guide_mean_probs.npy', guide_mean_probs)
np.save(DATA_DIR + 'change_mean_probs.npy', change_mean_probs)

print('Saved ensemble mean probabilities')

# Ensemble diversity diagnostics
for name, probs, logits in [
    ('GUIDE', guide_member_probs, guide_logits),
    ('CHANGE', change_member_probs, change_logits)
]:
    prob_corr = np.corrcoef(probs)
    prob_off = prob_corr[np.triu_indices_from(prob_corr, k=1)]

    logit_corr = np.corrcoef(logits)
    logit_off = logit_corr[np.triu_indices_from(logit_corr, k=1)]

    print(f'\n{name}')
    print(
        f'Probability correlation: '
        f'mean={prob_off.mean():.4f}, '
        f'min={prob_off.min():.4f}, '
        f'max={prob_off.max():.4f}'
    )
    print(
        f'Logit correlation: '
        f'mean={logit_off.mean():.4f}, '
        f'min={logit_off.min():.4f}, '
        f'max={logit_off.max():.4f}'
    )
    print(
        f'Mean ensemble variance: '
        f'{probs.var(axis=0).mean():.3e}'
    )

Saved ensemble mean probabilities

GUIDE
Probability correlation: mean=0.6164, min=0.4818, max=0.7918
Logit correlation: mean=0.6781, min=0.5667, max=0.8549
Mean ensemble variance: 4.138e-04

CHANGE
Probability correlation: mean=0.5981, min=0.4422, max=0.8089
Logit correlation: mean=0.7240, min=0.5855, max=0.8452
Mean ensemble variance: 5.124e-04


In [16]:
print('GUIDE-seq (within-dataset):')
print(f'AUROC: {roc_auc_score(y_guide_test, guide_mean_probs):.4f}')
print(f'AUPRC: {average_precision_score(y_guide_test, guide_mean_probs):.4f}')

print('\nCHANGE-seq (cross-dataset):')
print(f'AUROC: {roc_auc_score(y_change_test, change_mean_probs):.4f}')
print(f'AUPRC: {average_precision_score(y_change_test, change_mean_probs):.4f}')

print('\nPer-member performance:')

for name, member_probs, y in [
    ('GUIDE', guide_member_probs, y_guide_test),
    ('CHANGE', change_member_probs, y_change_test)
]:
    print(f'\n{name}')

    for i in range(member_probs.shape[0]):
        print(
            f'Member {i + 1}: '
            f'AUROC={roc_auc_score(y, member_probs[i]):.4f}, '
            f'AUPRC={average_precision_score(y, member_probs[i]):.4f}'
        )

GUIDE-seq (within-dataset):
AUROC: 0.8576
AUPRC: 0.0114

CHANGE-seq (cross-dataset):
AUROC: 0.8589
AUPRC: 0.1014

Per-member performance:

GUIDE
Member 1: AUROC=0.8539, AUPRC=0.0098
Member 2: AUROC=0.7969, AUPRC=0.0054
Member 3: AUROC=0.8663, AUPRC=0.0106
Member 4: AUROC=0.7555, AUPRC=0.0014
Member 5: AUROC=0.8504, AUPRC=0.0165

CHANGE
Member 1: AUROC=0.7990, AUPRC=0.0581
Member 2: AUROC=0.8348, AUPRC=0.0591
Member 3: AUROC=0.8557, AUPRC=0.1017
Member 4: AUROC=0.7470, AUPRC=0.0310
Member 5: AUROC=0.8512, AUPRC=0.1001


## DNABERT Embedding Extraction for Drift Detection

In [17]:
# Samples 5,000 examples from each test set for MMD
rng = np.random.default_rng(42)
n_sample = 5000

guide_idx = rng.choice(len(guide_test_dataset), size=n_sample, replace=False)
change_idx = rng.choice(len(change_test_dataset), size=n_sample, replace=False)

guide_mmd_dataset = Subset(guide_test_dataset, guide_idx)
change_mmd_dataset = Subset(change_test_dataset, change_idx)

print(f'GUIDE-seq MMD subset: {len(guide_mmd_dataset)} samples')
print(f'CHANGE-seq MMD subset: {len(change_mmd_dataset)} samples')

GUIDE-seq MMD subset: 5000 samples
CHANGE-seq MMD subset: 5000 samples


In [18]:
# Extract pretrained DNABERT embeddings with LoRA disabled
def extract_embeddings(model, dataset, batch_size=32):
    embeddings = []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.to(device)
    model.eval()
    
    with model.backbone.disable_adapter():
        with torch.inference_mode():
            for batch in loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                token_type_ids = batch['token_type_ids'].to(device)
                
                outputs = model.backbone(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )
                
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.append(cls_embeddings)
    
    model.to('cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return np.concatenate(embeddings, axis=0)

# Base DNABERT weights are shared across ensemble members
model = ensemble[0]

print('Extracting GUIDE-seq embeddings...')
guide_test_embeddings = extract_embeddings(model, guide_mmd_dataset)
print(f'Done. Shape: {guide_test_embeddings.shape}')

print('Extracting CHANGE-seq embeddings...')
change_test_embeddings = extract_embeddings(model, change_mmd_dataset)
print(f'Done. Shape: {change_test_embeddings.shape}')

Extracting GUIDE-seq embeddings...
Done. Shape: (5000, 768)
Extracting CHANGE-seq embeddings...
Done. Shape: (5000, 768)


In [19]:
# Saving to disk
np.save(DATA_DIR + 'guide_test_embeddings.npy', guide_test_embeddings)
np.save(DATA_DIR + 'change_test_embeddings.npy', change_test_embeddings)
print('Embeddings saved to', DATA_DIR)

Embeddings saved to data/processed/
